In [40]:
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, Subset
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import pickle
import os
from PIL import Image
from matplotlib.pyplot import GridSpec
import snntorch as snn
from snntorch import spikegen
from snntorch import surrogate
from snntorch import spikeplot

In [41]:
DATA_DIR = "../data"
DATASET_DIR = f"{DATA_DIR}/processed"

In [42]:
data = pd.read_parquet(f"{DATASET_DIR}/csecicids2018.parquet")

In [43]:
label_mapping = {
    'Benign': 'Benign',
    'Bot': 'Botnet',
    'FTP-BruteForce': 'Brute Force',
    'SSH-Bruteforce': 'Brute Force',
    'DDoS attacks-LOIC-HTTP': 'DDoS',
    'DDOS attack-LOIC-UDP': 'DDoS',
    'DDOS attack-HOIC': 'DDoS',
    'DoS attacks-GoldenEye': 'DoS',
    'DoS attacks-Slowloris': 'DoS',
    'DoS attacks-SlowHTTPTest': 'DoS',
    'DoS attacks-Hulk': 'DoS',
    'Infilteration': 'Infiltration',
    'Brute Force -Web': 'Brute Force',
    'Brute Force -XSS': 'Brute Force',
    'SQL Injection': 'Infiltration'  # Assuming SQL Injection is part of Infiltration
}

data["Label"] = data["Label"].map(label_mapping)

In [44]:
from sklearn.preprocessing import LabelEncoder

class CustomDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)
            label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
            return img, label

In [45]:
train_df = pd.read_csv(f"{DATASET_DIR}/train.csv")
test_df = pd.read_csv(f"{DATASET_DIR}/test.csv")
val_df = pd.read_csv(f"{DATASET_DIR}/val.csv")

In [46]:
LE = LabelEncoder()
LE.fit(train_df["Label"].unique())

LE.classes_

# swap classes in the label encoder
swapped_classes = LE.classes_.copy()
swapped_classes[0], swapped_classes[1] = swapped_classes[1], swapped_classes[0]

LE.classes_ = swapped_classes

train_df_encoded = train_df.copy()
train_df_encoded["Label"] = LE.transform(train_df_encoded["Label"])

val_df_encoded = val_df.copy()
val_df_encoded["Label"] = LE.transform(val_df_encoded["Label"])

train_df_encoded.head()

,Image,Label
0,image_1119705.png,1
1,image_2975672.png,1
2,image_6489183.png,1
3,image_2675390.png,1
4,image_5559062.png,1


In [47]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    # transforms.RandomHorizontalFlip(),  # randomly flip images horizontally
    # transforms.RandomRotation(10),      # slight random rotation
    # transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # random translation
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.5], std=[0.5])  # normalize to [-1, 1] range
])

# Keep validation transforms simple - just basic preprocessing
val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = CustomDataset(train_df_encoded, f"{DATASET_DIR}/images", transform=train_transform)
val_dataset = CustomDataset(val_df_encoded, f"{DATASET_DIR}/images", transform=val_transform)

In [48]:
from torch.utils.data import WeightedRandomSampler

labels = train_df_encoded["Label"].values
class_weights = 1 / np.bincount(labels)
sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

In [49]:
BATCH_SIZE = 40
train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_data_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [66]:
class ConvSpikingLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding='valid', 
                 threshold=0.3, alpha=0.5, beta=0.5, time_steps=4):
        super(ConvSpikingLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, 
                            padding=0 if padding=='valid' else padding)
        
        # Initialize the spiking neuron parameters
        self.lif = snn.Leaky(beta=beta,                     # Decay rate
                            threshold=threshold,             # Firing threshold
                            reset_mechanism="subtract",      # Subtract threshold when spiking
                            spike_grad=surrogate.fast_sigmoid(slope=25), # Surrogate gradient
                            # alpha=alpha                    # Membrane potential reset scaling factor
                            )                     
        
        self.time_steps = time_steps
        self.alpha = alpha

    def forward(self, x):
        # Initialize membrane potential
        mem = torch.zeros(x.shape[0], self.conv.out_channels, 
                         ((x.shape[2] - self.conv.kernel_size[0]) // self.conv.stride[0]) + 1,
                         ((x.shape[3] - self.conv.kernel_size[1]) // self.conv.stride[1]) + 1,
                         device=x.device)
        
        # Initialize spike accumulator
        spk_rec = []
        mem_rec = []

        # Simulate for t timesteps
        for t in range(self.time_steps):
            cur_x = x[:, :, :, :, t] if len(x.shape) > 4 else x
            conv_x = self.conv(cur_x)
            spk, mem = self.lif(conv_x, mem)

            # manually apply membrane potential reset scaling factor
            mem = mem * self.alpha

            spk_rec.append(spk)
            mem_rec.append(mem)

        return torch.stack(spk_rec, dim=-1), torch.stack(mem_rec, dim=-1)

class TimeValEncoder(nn.Module):
    def __init__(self, time_steps):
        super(TimeValEncoder, self).__init__()
        self.weights = self._generate_weights(time_steps)
    
    def _generate_weights(self, n):
        M_t = 1
        list_n = []
        for _ in range(n):
            list_n.append(M_t)
            M_t *= 2
        list_n.reverse()
        
        list_n = torch.tensor(list_n, dtype=torch.float32)
        return list_n / list_n.sum()
    
    def forward(self, spikes):
        return (spikes * self.weights.to(spikes.device)).sum(dim=-1)

class BinarySNNClassifier(nn.Module):
    def __init__(self, time_steps=4):
        super(BinarySNNClassifier, self).__init__()
        
        # Reduced number of channels in intermediate layers since we only need 2 classes
        self.conv1 = ConvSpikingLayer(
            in_channels=1,
            out_channels=8,  # Reduced from 16
            kernel_size=4,
            stride=4,
            threshold=0.3,
            time_steps=time_steps
        )
        
        self.conv2 = ConvSpikingLayer(
            in_channels=8,   # Reduced from 16
            out_channels=16,  # Reduced from 32
            kernel_size=2,
            stride=2,
            threshold=0.3,
            time_steps=time_steps
        )
        
        self.encoder = TimeValEncoder(time_steps)
        
        # Final layer outputs single value for binary classification
        self.fc = nn.Linear(16 * 4 * 4, 1)  # Reduced from 32 channels to 16
        self.sigmoid = nn.Sigmoid()  # Add sigmoid for binary classification
    
    def forward(self, x):
        spk1, mem1 = self.conv1(x)
        spk2, mem2 = self.conv2(spk1)
        encoded = self.encoder(spk2)
        flat = encoded.flatten(1)
        out = self.fc(flat)
        return self.sigmoid(out)  # Return probability between 0 and 1

# class BinaryCustomLoss(nn.Module):
#     def __init__(self):
#         super(BinaryCustomLoss, self).__init__()
#         self.bce = nn.BCELoss()
    
#     def forward(self, predict, target):
#         # Simplified loss for binary classification
#         bce_loss = self.bce(predict, target)
        
#         # Add ranking component
#         confidence_penalty = torch.mean(torch.abs(predict - 0.5))
        
#         return bce_loss - 0.1 * confidence_penalty

class BinaryCustomLoss(nn.Module):
    def __init__(self, pos_weight=5.0):
        super(BinaryCustomLoss, self).__init__()
        self.pos_weight = pos_weight
        self.eps = 1e-8  # for numerical stability

    def forward(self, predict, target):
        # Manual binary cross entropy with pos_weight
        loss = - (self.pos_weight * target * torch.log(predict + self.eps) + 
                  (1 - target) * torch.log(1 - predict + self.eps))
        bce_loss = torch.mean(loss)
        
        # Add ranking component (unchanged)
        confidence_penalty = torch.mean(torch.abs(predict - 0.5))
        
        return bce_loss - 0.1 * confidence_penalty
    

class SNNClassifier(nn.Module):
    def __init__(self, num_classes=2, time_steps=4):
        super(SNNClassifier, self).__init__()
        
        # First convolutional spiking layer
        # Input: (batch, 1, 32, 32)
        # Output: (batch, 16, 8, 8)
        self.conv1 = ConvSpikingLayer(
            in_channels=1,
            out_channels=16,
            kernel_size=4,
            stride=4,
            threshold=0.3,
            time_steps=time_steps
        )
        
        # Second convolutional spiking layer
        # Input: (batch, 16, 8, 8)
        # Output: (batch, 32, 4, 4)
        self.conv2 = ConvSpikingLayer(
            in_channels=16,
            out_channels=32,
            kernel_size=2,
            stride=2,
            threshold=0.3,
            time_steps=time_steps
        )
        
        # Time-value encoder
        self.encoder = TimeValEncoder(time_steps)
        
        # Final linear layer
        self.fc = nn.Linear(32 * 4 * 4, num_classes)
    
    def forward(self, x):
        # First convolutional layer
        spk1, mem1 = self.conv1(x)
        
        # Second convolutional layer
        spk2, mem2 = self.conv2(spk1)
        
        # Encode spikes using time-value encoding
        encoded = self.encoder(spk2)
        
        # Flatten and pass through linear layer
        flat = encoded.flatten(1)
        out = self.fc(flat)
        
        return out

# class CustomLoss(nn.Module):
#     def __init__(self, n_classes):
#         super(CustomLoss, self).__init__()
#         self.n_classes = n_classes
    
#     def forward(self, predict, target):
#         # Implementation of Algorithm 2
#         # Get correlation between prediction and target
#         cor = (predict * target).max(1)[0]
        
#         # Get maximum prediction values
#         pre = predict.max(1)[0]
        
#         # Get ranking of correct classification
#         val = target.max(1)[0]
#         idx = target.max(1)[1]
#         ids = torch.zeros_like(val)
        
#         for i in range(len(idx)):
#             val[i] = predict[i, idx[i]]
#             ids[i] = (predict[i] > val[i]).float().sum()
        
#         alpha = pre - cor
#         beta = 1 - cor
        
#         return torch.mean(self.n_classes * alpha + (ids + 1) * beta)


class CustomLoss(nn.Module):
    def __init__(self, n_classes, class_weights=None):
        super(CustomLoss, self).__init__()
        self.n_classes = n_classes
        # Expect class_weights to be a list or tensor of length n_classes.
        # If provided, convert to a tensor.
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None
    
    def forward(self, predict, target):
        # Get correlation between prediction and target
        cor = (predict * target).max(1)[0]
        
        # Get maximum prediction values
        pre = predict.max(1)[0]
        
        # Get ranking of correct classification
        val = target.max(1)[0]
        idx = target.max(1)[1]
        ids = torch.zeros_like(val)
        
        for i in range(len(idx)):
            val[i] = predict[i, idx[i]]
            ids[i] = (predict[i] > val[i]).float().sum()
        
        alpha = pre - cor
        beta = 1 - cor
        
        # Compute base loss per sample
        loss = self.n_classes * alpha + (ids + 1) * beta
        
        # If class weights are provided, weight each sample's loss by the corresponding class weight.
        if self.class_weights is not None:
            # idx contains the true class index for each sample.
            weights = self.class_weights[idx].to(predict.device)
            loss = loss * weights
        
        return torch.mean(loss)


In [78]:
def train_binary_model(model, train_loader, val_loader, num_epochs=50, device='cuda', early_stop=10, 
                       model_savepath=None, binary=True, learning_rate=0.001, weight_decay=0, n_classes=None, weights=None):
    import torch
    import numpy as np
    from tqdm import tqdm
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    # Set the appropriate loss function based on the model type
    if binary:
        loss_fn = BinaryCustomLoss(pos_weight=weights)
    else:
        loss_fn = CustomLoss(n_classes, class_weights=weights)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Track best model and metrics
    best_val_loss = float('inf')
    best_epoch = 0
    best_model = None
    early_stop_counter = 0
    best_threshold = 0.5
    
    # History dictionary to track metrics
    history = {
        "train_loss": [], "val_loss": [],
        "metrics": {
            "train": {"accuracy": [], "precision": [], "recall": [], "f1": []},
            "val": {"accuracy": [], "precision": [], "recall": [], "f1": []}
        }
    }
    
    def adjust_learning_rate(optimizer, epoch, T_max=50, initial_lr=learning_rate):
        if epoch == 0:
            lr = initial_lr
        else:
            lr = 0.5 * initial_lr * (1 + torch.cos(torch.tensor(epoch / T_max * torch.pi)))
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
    
    def calculate_metrics(true_labels, pred_labels, pred_probs=None):
        # Handle multi-class case differently than binary case
        if not binary:
            # For multi-class, calculate only accuracy
            metrics = {
                "accuracy": accuracy_score(true_labels, pred_labels),
                "precision": 0,  # Placeholder
                "recall": 0,     # Placeholder
                "f1": 0          # Placeholder
            }
            metrics["best_threshold"] = 0.5  # Not applicable for multi-class
            return metrics
        
        # Binary classification metrics
        metrics = {
            "accuracy": accuracy_score(true_labels, pred_labels),
            "precision": precision_score(true_labels, pred_labels, zero_division=0),
            "recall": recall_score(true_labels, pred_labels, zero_division=0),
            "f1": f1_score(true_labels, pred_labels, zero_division=0)
        }
        
        # Find optimal threshold if probabilities are provided for binary case
        if pred_probs is not None:
            thresholds = np.arange(0.1, 0.9, 0.05)
            best_f1 = 0
            best_thresh = 0.5
            
            for threshold in thresholds:
                thresh_preds = (np.array(pred_probs) > threshold).astype(np.int8)
                f1 = f1_score(true_labels, thresh_preds, zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1
                    best_thresh = threshold
            
            metrics["best_threshold"] = best_thresh
        
        return metrics
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 50)
        
        # Adjust learning rate
        adjust_learning_rate(optimizer, epoch)
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_preds = []
        train_true = []
        train_probs = []
        
        for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc="Training Batches")):
            data = data.to(device)
            
            if binary:
                # For binary classification
                target = target.float().to(device)
                if len(target.shape) == 1:
                    target = target.view(-1, 1)
            else:
                # For multi-class classification
                target = target.long().to(device)
            
            optimizer.zero_grad()
            output = model(data)
            
            # Handle binary vs multi-class loss calculation
            if binary:
                loss = loss_fn(output, target)
            else:
                # For multi-class CustomLoss
                # Convert targets to one-hot for CustomLoss
                target_onehot = torch.zeros(target.size(0), n_classes, device=device)
                target_onehot.scatter_(1, target.view(-1, 1), 1)
                loss = loss_fn(output, target_onehot)
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item() * data.size(0)
            
            # Get predictions
            if binary:
                # Binary case - model already applies sigmoid
                probs = output.detach().cpu().numpy()
                preds = (probs > 0.5).astype(np.int8)
                true = target.detach().cpu().numpy()
            else:
                # Multi-class case
                _, predicted = torch.max(output.data, 1)
                preds = predicted.detach().cpu().numpy()
                true = target.detach().cpu().numpy()
                probs = None  # Not using probabilities for multi-class here
            
            # Flatten arrays in case of multi-dimensional outputs
            if len(preds.shape) > 1:
                preds = preds.flatten()
            if len(true.shape) > 1:
                true = true.flatten()
            if binary and probs is not None and len(probs.shape) > 1:
                probs = probs.flatten()
            
            train_preds.extend(preds)
            train_true.extend(true)
            if binary and probs is not None:
                train_probs.extend(probs)
            
            if batch_idx % 100 == 0:
                print(f'Train Epoch: {epoch+1} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                      f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')
        
        train_loss /= len(train_loader.dataset)
        train_metrics = calculate_metrics(train_true, train_preds, train_probs if binary else None)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_true = []
        val_probs = []
        
        with torch.no_grad():
            for data, target in tqdm(val_loader, desc="Validation Batches"):
                data = data.to(device)
                
                if binary:
                    # For binary classification
                    target = target.float().to(device)
                    if len(target.shape) == 1:
                        target = target.view(-1, 1)
                else:
                    # For multi-class classification
                    target = target.long().to(device)
                
                output = model(data)
                
                # Handle binary vs multi-class loss calculation
                if binary:
                    loss = loss_fn(output, target)
                else:
                    # For multi-class CustomLoss
                    # Convert targets to one-hot for CustomLoss
                    target_onehot = torch.zeros(target.size(0), n_classes, device=device)
                    target_onehot.scatter_(1, target.view(-1, 1), 1)
                    loss = loss_fn(output, target_onehot)
                
                val_loss += loss.item() * data.size(0)
                
                # Get predictions
                if binary:
                    # Binary case - model already applies sigmoid
                    probs = output.detach().cpu().numpy()
                    preds = (probs > 0.5).astype(np.int8)
                    true = target.detach().cpu().numpy()
                else:
                    # Multi-class case
                    _, predicted = torch.max(output.data, 1)
                    preds = predicted.detach().cpu().numpy()
                    true = target.detach().cpu().numpy()
                    probs = None  # Not using probabilities for multi-class here
                
                # Flatten arrays in case of multi-dimensional outputs
                if len(preds.shape) > 1:
                    preds = preds.flatten()
                if len(true.shape) > 1:
                    true = true.flatten()
                if binary and probs is not None and len(probs.shape) > 1:
                    probs = probs.flatten()
                
                val_preds.extend(preds)
                val_true.extend(true)
                if binary and probs is not None:
                    val_probs.extend(probs)
        
        val_loss /= len(val_loader.dataset)
        val_metrics = calculate_metrics(val_true, val_preds, val_probs if binary else None)
        
        # For binary models, get the best threshold from validation
        if binary:
            epoch_best_threshold = val_metrics["best_threshold"]
        else:
            epoch_best_threshold = 0.5  # Not applicable for multi-class
        
        # Update history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        
        for metric in ["accuracy", "precision", "recall", "f1"]:
            history["metrics"]["train"][metric].append(train_metrics[metric])
            history["metrics"]["val"][metric].append(val_metrics[metric])
        
        # Print metrics
        print(f"\nLoss - Train: {train_loss:.4f}, Val: {val_loss:.4f}")
        print(f"Accuracy - Train: {train_metrics['accuracy']:.4f}, Val: {val_metrics['accuracy']:.4f}")
        
        if binary:
            print(f"Best Threshold for this epoch: {epoch_best_threshold:.2f}")
            print(f"Precision - Train: {train_metrics['precision']:.4f}, Val: {val_metrics['precision']:.4f}")
            print(f"Recall - Train: {train_metrics['recall']:.4f}, Val: {val_metrics['recall']:.4f}")
            print(f"F1 Score - Train: {train_metrics['f1']:.4f}, Val: {val_metrics['f1']:.4f}")
        
        # Early stopping with model saving
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_model = model.state_dict()
            best_threshold = epoch_best_threshold if binary else 0.5
            if model_savepath:
                torch.save(model.state_dict(), model_savepath)
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if early_stop_counter >= early_stop:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
    
    print(f"\nBest model was from epoch {best_epoch+1} with validation loss {best_val_loss:.4f}")
    if binary:
        print(f"Best threshold: {best_threshold:.2f}")
    
    
    return best_model, history, best_threshold

In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [77]:
model_savepath = f"../models/checkpoints"

pathname_map = {
    'BinarySNN': "bsnn",
    'SNN': "snn",
}


benign_count = train_df_encoded["Label"].value_counts()[1]
malicious_count = train_df_encoded["Label"].value_counts().sum() - benign_count

pos_weight = torch.tensor([malicious_count / benign_count], dtype=torch.float32).to(device)

pos_weight


tensor([5.], device='cuda:0')

In [76]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

# Compute class weights for CustomLoss
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_df_encoded["Label"]), y=train_df_encoded["Label"])

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

In [79]:
for model_class in [SNNClassifier, BinarySNNClassifier]:
    model_name = model_class.__name__
    print(f"Training {model_name}")
    
    model = model_class().to(device)
    
    if model_name == "BinarySNNClassifier":
        model_savepath = f"{model_savepath}/{pathname_map["BinarySNN"]}.pth"
        best_model, history, best_threshold = train_binary_model(
            model, train_data_loader, val_data_loader, num_epochs=50, device=device, early_stop=50, 
            model_savepath=model_savepath, binary=True, learning_rate=0.001, weight_decay=0, n_classes=None, weights=pos_weight.item()
        )
    else:
        model_savepath = f"{model_savepath}/{pathname_map["SNN"]}.pth"
        best_model, history = train_binary_model(
            model, train_data_loader, val_data_loader, num_epochs=50, device=device, early_stop=50, 
            model_savepath=model_savepath, binary=False, learning_rate=0.001, weight_decay=0, n_classes=2, weights=class_weights
        )
    

C:\Users\para_\AppData\Local\Temp\ipykernel_29292\2022083545.py:217: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.class_weights = torch.tensor(class_weights, dtype=torch.float32)


Training SNNClassifier

Epoch 1/50
--------------------------------------------------


Training Batches:   0%|          | 0/750 [00:00<?, ?it/s]

Training Batches:   0%|          | 3/750 [00:00<01:26,  8.63it/s]

Train Epoch: 1 [0/30000 (0%)]	Loss: 1.909983


...

Train Epoch: 49 [24000/30000 (80%)]	Loss: -18895.958984


Training Batches:  94%|█████████▍| 704/750 [00:42<00:02, 18.70it/s]

Train Epoch: 49 [28000/30000 (93%)]	Loss: -17681.587891


Validation Batches:  38%|███▊      | 283/750 [00:13<00:22, 20.54it/s]


KeyboardInterrupt: 